In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
from google.colab import files

uploaded = files.upload()

Saving Online Retail.xlsx to Online Retail.xlsx


In [3]:
df = pd.read_excel("Online Retail.xlsx")

df = df.drop_duplicates().copy()

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"], errors="coerce"
)

data = df[
    (df["UnitPrice"] > 0) &
    (df["Quantity"] > 0) &
    (df["InvoiceDate"].notna()) &
    (df["StockCode"].notna())
].copy()

data["Month"] = data["InvoiceDate"].dt.month
data["Hour"] = data["InvoiceDate"].dt.hour
data["DayOfWeek"] = data["InvoiceDate"].dt.dayofweek

print("Dataset shape:", data.shape)
data.head()

Dataset shape: (524878, 11)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Month,Hour,DayOfWeek
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,12,8,2
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,12,8,2
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,12,8,2
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,12,8,2
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,12,8,2


In [4]:
dml_sample = data[
    ["UnitPrice", "Quantity", "Month", "Hour", "DayOfWeek"]
].dropna().copy()

if len(dml_sample) > 50000:
    dml_sample = dml_sample.sample(
        n=50000, random_state=42
    )

T = dml_sample["UnitPrice"].to_numpy()
Y = dml_sample["Quantity"].to_numpy()

X = dml_sample[
    ["Month", "Hour", "DayOfWeek"]
].to_numpy()

print("Sample size:", len(dml_sample))
print("Treatment: UnitPrice")
print("Outcome: Quantity")
print("Control features: Month, Hour, DayOfWeek")

Sample size: 50000
Treatment: UnitPrice
Outcome: Quantity
Control features: Month, Hour, DayOfWeek


In [5]:
kf = KFold(n_splits=2, shuffle=True, random_state=42)

T_residual = np.zeros(len(T))
Y_residual = np.zeros(len(Y))

for train_idx, test_idx in kf.split(X):
    model_t = RandomForestRegressor(
        n_estimators=50,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )

    model_y = RandomForestRegressor(
        n_estimators=50,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )

    model_t.fit(X[train_idx], T[train_idx])
    model_y.fit(X[train_idx], Y[train_idx])

    T_residual[test_idx] = (
        T[test_idx] - model_t.predict(X[test_idx])
    )

    Y_residual[test_idx] = (
        Y[test_idx] - model_y.predict(X[test_idx])
    )

final_model = LinearRegression(fit_intercept=False)
final_model.fit(T_residual.reshape(-1, 1), Y_residual)

theta = final_model.coef_[0]

print("DML prototype coefficient:", round(theta, 6))
print("Treatment residual std:", round(T_residual.std(), 4))

DML prototype coefficient: -0.009319
Treatment residual std: 48.8526


In [6]:
report = pd.DataFrame({
    "Metric": [
        "Sample size",
        "Treatment",
        "Outcome",
        "Control features",
        "DML prototype coefficient",
        "Treatment residual standard deviation"
    ],
    "Value": [
        len(dml_sample),
        "UnitPrice",
        "Quantity",
        "Month, Hour, DayOfWeek",
        theta,
        T_residual.std()
    ]
})

display(report)

report.to_csv(
    "day12_dml_prototype_summary.csv",
    index=False
)

print("Report saved successfully!")

,Metric,Value
0,Sample size,50000
1,Treatment,UnitPrice
2,Outcome,Quantity
3,Control features,"Month, Hour, DayOfWeek"
4,DML prototype coefficient,-0.009319
5,Treatment residual standard deviation,48.8526


Report saved successfully!


In [7]:
from google.colab import files

files.download("day12_dml_prototype_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>